# TurboQuant Testing Notebook

Tests for `experiment1_turboquant`: 3-bit weight compression via randomized Hadamard rotation + Lloyd-Max quantization.

**Sections:**
1. Setup (clone repo, install deps)
2. Core algorithm unit tests
3. Layer wrapper tests (TQLinear, TQConv2d)
4. PINN model quantization + evaluation
5. Compression report + visualizations

## 1. Setup

In [ ]:
# Clone the repo (run once in Colab)
import os

REPO_URL = "https://github.com/YOUR_USERNAME/ECE228_ML_for_Physical_Applications.git"  # update this
REPO_DIR = "/content/ECE228_ML_for_Physical_Applications"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned.")

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [1]:
!pip install -q torch torchvision numpy scipy opencv-python-headless Pillow matplotlib tqdm

In [6]:
import sys
import os

REPO_DIR = "/content/ECE228_ML_for_Physical_Applications"
TQ_DIR   = os.path.join(REPO_DIR, "experiment1_turboquant")
PINN_DIR = os.path.join(REPO_DIR, "PINN_channel-estimation-main")

for p in [TQ_DIR, PINN_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Model.py uses find_in_map which resolves paths relative to PINN_DIR
os.chdir(PINN_DIR)

print("TQ_DIR :", TQ_DIR)
print("PINN_DIR:", PINN_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/content/ECE228_ML_for_Physical_Applications/PINN_channel-estimation-main'

In [ ]:
import math
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from turboquant import (
    TurboQuantTensor,
    fast_hadamard_transform,
    random_rotation,
    inverse_rotation,
    quantize_3bit,
    dequantize_3bit,
    LLOYD_MAX_3BIT,
    LLOYD_MAX_3BIT_DISTORTION_RATIO,
)
from tq_layers import TQLinear, TQConv2d, TQConvTranspose2d, _safe_block_dim
from quantize_pinn import quantize_model, model_size_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Core Algorithm Tests

In [ ]:
# Test 1 & 2: Fast Hadamard Transform — H^2 = I
print("=== Fast Hadamard Transform ===")

errors = []
for d in [4, 8, 16, 64, 128, 256]:
    x = torch.randn(d, d)
    x_roundtrip = fast_hadamard_transform(fast_hadamard_transform(x))
    err = (x_roundtrip - x).abs().max().item()
    errors.append((d, err))
    status = "PASS" if err < 1e-5 else "FAIL"
    print(f"  d={d:4d}: max_err={err:.2e}  [{status}]")

dims, errs = zip(*errors)
plt.figure(figsize=(7, 3))
plt.bar([str(d) for d in dims], errs)
plt.axhline(1e-5, color='r', linestyle='--', label='threshold 1e-5')
plt.yscale('log')
plt.xlabel('d (last dim)')
plt.ylabel('max |H(H(x)) - x|')
plt.title('FHT self-inverse error (should be < 1e-5)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Test 3: Random rotation invertibility
print("=== Rotation Invertibility ===")
torch.manual_seed(42)

all_pass = True
for d in [8, 64, 128]:
    for seed in [0, 42, 12345]:
        x = torch.randn(10, d)
        y = random_rotation(x, seed=seed, n_blocks=3)
        x_recon = inverse_rotation(y, seed=seed, n_blocks=3)
        err = (x_recon - x).abs().max().item()
        ok = err < 1e-4
        if not ok:
            all_pass = False
        print(f"  d={d:3d}, seed={seed:6d}: max_err={err:.2e}  [{'PASS' if ok else 'FAIL'}]")

print(f"\nRotation invertibility: {'ALL PASS' if all_pass else 'SOME FAILED'}")

In [ ]:
# Test 4: Lloyd-Max 3-bit distortion
print("=== Lloyd-Max 3-bit Distortion ===")
torch.manual_seed(3)

n = 200_000
x = torch.randn(n)
idx = quantize_3bit(x)
x_hat = dequantize_3bit(idx)
mse = ((x - x_hat) ** 2).mean().item()
variance = x.var().item()
ratio = mse / variance

print(f"  MSE:              {mse:.6f}")
print(f"  Variance:         {variance:.6f}")
print(f"  MSE/Var ratio:    {ratio:.6f}")
print(f"  Theoretical:      {LLOYD_MAX_3BIT_DISTORTION_RATIO:.6f}")
ok = 0.005 < ratio < 0.05
print(f"  Status: {'PASS' if ok else 'FAIL'} (expected 0.005-0.05)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_np = x[:5000].numpy()
xhat_np = x_hat[:5000].numpy()

axes[0].hist(x_np, bins=80, alpha=0.6, label='original N(0,1)', density=True)
axes[0].hist(xhat_np, bins=80, alpha=0.6, label='quantized', density=True)
for c in LLOYD_MAX_3BIT.numpy():
    axes[0].axvline(c, color='r', linewidth=0.8, alpha=0.7)
axes[0].set_title('Lloyd-Max 3-bit centroids (red lines)')
axes[0].legend()

axes[1].scatter(x_np[:500], xhat_np[:500], s=2, alpha=0.4)
axes[1].plot([-3, 3], [-3, 3], 'r--', label='ideal')
axes[1].set_xlabel('original')
axes[1].set_ylabel('quantized')
axes[1].set_title('Original vs quantized (first 500 samples)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Rotation spreads outliers (key property of TurboQuant)
print("=== Outlier Spreading ===")
torch.manual_seed(10)
d = 128
x = torch.zeros(1, d)
x[0, 0] = 1000.0
x[0, 1:] = torch.randn(d - 1) * 1.0

y = random_rotation(x, seed=42, n_blocks=3)

ratio_before = (x.abs().max() / x.abs().mean()).item()
ratio_after  = (y.abs().max() / y.abs().mean()).item()
print(f"  max/mean ratio BEFORE rotation: {ratio_before:.1f}")
print(f"  max/mean ratio AFTER  rotation: {ratio_after:.1f}")
print(f"  Spread factor: {ratio_before / ratio_after:.1f}x reduction")
print(f"  Status: {'PASS' if ratio_after < ratio_before * 0.1 else 'FAIL'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(range(d), x[0].numpy(), width=1.0)
axes[0].set_title('Before rotation (outlier at index 0)')
axes[0].set_xlabel('coordinate index')
axes[1].bar(range(d), y[0].numpy(), width=1.0)
axes[1].set_title('After rotation (energy spread uniformly)')
axes[1].set_xlabel('coordinate index')
plt.tight_layout()
plt.show()

## 3. TurboQuantTensor Compress/Decompress

In [ ]:
# Test 5 & 6: Roundtrip + compression ratio
print("=== TurboQuantTensor Roundtrip + Compression ===")
torch.manual_seed(4)

configs = [
    ((256, 256), 128, "Linear weight 256x256"),
    ((64, 32, 3, 3), 64,  "Conv2d 64x32x3x3"),
    ((32, 64, 3, 3), 64,  "ConvTranspose2d 32x64x3x3"),
    ((100, 100), 64,       "Non-pow2 total 10000"),
    ((128,), 128,          "1-D weight 128"),
]

for shape, bd, label in configs:
    W = torch.randn(*shape) * 0.02
    tqt = TurboQuantTensor.compress(W, block_dim=bd, seed=7)
    W_hat = tqt.decompress(dtype=torch.float32)

    assert W_hat.shape == W.shape, f"Shape mismatch: {W_hat.shape} vs {W.shape}"
    mse   = ((W_hat - W) ** 2).mean().item()
    var   = W.var().item()
    ratio_mse = mse / max(var, 1e-12)
    ratio_comp = tqt.compression_ratio()
    ok = ratio_mse < 0.15
    print(f"  {label}")
    print(f"    MSE/Var={ratio_mse:.4f}  Compression={ratio_comp:.1f}x  [{'PASS' if ok else 'FAIL'}]")

W_big = torch.randn(256, 256)
tqt_big = TurboQuantTensor.compress(W_big, block_dim=128, seed=1)
cr = tqt_big.compression_ratio()
print(f"\nCompression ratio (256x256, bd=128): {cr:.2f}x  (expected ~10.2x)")
print(f"Status: {'PASS' if 8.0 <= cr <= 12.0 else 'FAIL'}")

In [ ]:
# Sweep block_dim — MSE/Var vs compression tradeoff
torch.manual_seed(99)
W = torch.randn(512, 512) * 0.02
block_dims = [8, 16, 32, 64, 128, 256, 512]

mse_ratios, comp_ratios = [], []
for bd in block_dims:
    tqt = TurboQuantTensor.compress(W, block_dim=bd, seed=7)
    W_hat = tqt.decompress()
    mse_ratios.append(((W_hat - W)**2).mean().item() / W.var().item())
    comp_ratios.append(tqt.compression_ratio())

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(block_dims, mse_ratios, 'b-o', label='MSE/Var (lower=better)')
ax2.plot(block_dims, comp_ratios, 'r-s', label='Compression ratio (higher=better)')
ax1.set_xlabel('block_dim')
ax1.set_ylabel('MSE/Var', color='b')
ax2.set_ylabel('Compression ratio (x)', color='r')
ax1.set_xscale('log', base=2)
ax1.set_yscale('log')
ax1.set_title('TurboQuant: MSE quality vs compression tradeoff')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

## 4. Layer Wrappers

In [ ]:
print("=== TQLinear ===")
torch.manual_seed(6)

lin = nn.Linear(256, 128, bias=True)
tq_lin = TQLinear(lin, block_dim=128, seed=99)
print(tq_lin)

x = torch.randn(4, 256)
with torch.no_grad():
    y_fp32 = lin(x)
    y_tq   = tq_lin(x)

diff     = (y_tq - y_fp32).abs().mean().item()
scale    = y_fp32.abs().mean().item()
rel_err  = diff / max(scale, 1e-12)

print(f"  Mean absolute diff: {diff:.6f}")
print(f"  Mean absolute FP32: {scale:.6f}")
print(f"  Relative error:     {rel_err:.4f}")
print(f"  Bias preserved:     {(tq_lin.bias.data - lin.bias.data).abs().max().item():.2e}")
print(f"  Status: {'PASS' if rel_err < 0.5 and diff > 0 else 'FAIL'}")

In [ ]:
print("=== TQConv2d ===")
torch.manual_seed(7)

conv = nn.Conv2d(32, 64, kernel_size=3, padding=1)
tq_conv = TQConv2d(conv, block_dim=64, seed=77)
print(tq_conv)

x = torch.randn(2, 32, 16, 16)
with torch.no_grad():
    y_fp32 = conv(x)
    y_tq   = tq_conv(x)

rel_err = (y_tq - y_fp32).abs().mean() / y_fp32.abs().mean().clamp(min=1e-8)
print(f"  Output shape: {y_tq.shape} (matches: {y_fp32.shape == y_tq.shape})")
print(f"  Relative error: {rel_err.item():.4f}  Status: {'PASS' if rel_err < 0.5 else 'FAIL'}")

print("\n=== TQConvTranspose2d ===")
convt = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
tq_convt = TQConvTranspose2d(convt, block_dim=64, seed=88)
print(tq_convt)

x = torch.randn(2, 64, 8, 8)
with torch.no_grad():
    y_fp32 = convt(x)
    y_tq   = tq_convt(x)

rel_err = (y_tq - y_fp32).abs().mean() / y_fp32.abs().mean().clamp(min=1e-8)
print(f"  Output shape: {y_tq.shape}")
print(f"  Relative error: {rel_err.item():.4f}  Status: {'PASS' if rel_err < 0.5 else 'FAIL'}")

In [ ]:
print("=== _safe_block_dim ===")
cases = [
    (108,   128, 64,  "Conv 12ch filter (12*9=108 elements)"),
    (9,     128, 8,   "Tiny 9-element weight"),
    (18432, 128, 128, "Large 18432-element weight"),
    (1,     128, 1,   "Single element"),
]
for numel, preferred, expected, label in cases:
    got = _safe_block_dim(numel, preferred)
    ok = got == expected
    print(f"  numel={numel:6d}, preferred={preferred}: got={got:3d} expected={expected:3d}  [{label}]  [{'PASS' if ok else 'FAIL'}]")

## 5. PINN Model Quantization

In [ ]:
print("=== PINN Model Build ===")
try:
    from Model import ImprovedPhysicsInformedUNet
    print("Model.py imported successfully.")
except ImportError as e:
    print(f"IMPORT ERROR: {e}")
    print("Make sure PINN_channel-estimation-main/ is in the repo and paths are set in section 1.")
    raise

torch.manual_seed(9)
model_fp32 = ImprovedPhysicsInformedUNet(
    channel_shape=(32, 4, 576),
    rss_size=64,
    latent_dim=256,
    use_dbm_values=True,
).eval()

n_params = sum(p.numel() for p in model_fp32.parameters())
fp32_mb  = sum(p.numel() * p.element_size() for p in model_fp32.parameters()) / 1e6
print(f"  Parameters:      {n_params:,}")
print(f"  FP32 size:       {fp32_mb:.1f} MB")
print(f"  Linear:          {sum(1 for m in model_fp32.modules() if isinstance(m, nn.Linear))}")
print(f"  Conv2d:          {sum(1 for m in model_fp32.modules() if isinstance(m, nn.Conv2d))}")
print(f"  ConvTranspose2d: {sum(1 for m in model_fp32.modules() if isinstance(m, nn.ConvTranspose2d))}")

In [ ]:
print("=== Quantizing PINN Model ===")
t0 = time.perf_counter()
model_q = quantize_model(
    model_fp32,
    block_dim_linear=128,
    block_dim_conv=64,
    inplace=False,
).eval()
t_quant = time.perf_counter() - t0
print(f"  Quantization time: {t_quant:.2f} s")

n_tq_lin  = sum(1 for m in model_q.modules() if isinstance(m, TQLinear))
n_tq_conv = sum(1 for m in model_q.modules() if isinstance(m, TQConv2d))
n_tq_convt= sum(1 for m in model_q.modules() if isinstance(m, TQConvTranspose2d))
print(f"  TQLinear:          {n_tq_lin}")
print(f"  TQConv2d:          {n_tq_conv}")
print(f"  TQConvTranspose2d: {n_tq_convt}")

assert n_tq_lin > 0 and n_tq_conv > 0, "No TQ layers found!"
print("  [PASS] TQ layers found.")

In [ ]:
print(model_size_report(model_fp32, model_q))

In [ ]:
print("=== Forward Pass Comparison ===")
torch.manual_seed(42)
smomp   = torch.randn(2, 32, 4, 576) * 0.1
rss_map = torch.rand(2, 2, 64, 64) * 2 - 1

with torch.no_grad():
    t0 = time.perf_counter()
    pred_fp32 = model_fp32(smomp, rss_map)
    t_fp32 = time.perf_counter() - t0

    t0 = time.perf_counter()
    pred_tq = model_q(smomp, rss_map)
    t_tq = time.perf_counter() - t0

assert torch.isfinite(pred_tq).all(), "TQ output has NaN/Inf!"
assert pred_fp32.shape == pred_tq.shape == (2, 32, 4, 576)

diff_mse = ((pred_tq - pred_fp32) ** 2).mean().item()
fp32_power = (pred_fp32 ** 2).mean().item()
output_distortion_db = 10 * math.log10(diff_mse / max(fp32_power, 1e-12))

print(f"  Output shape:            {pred_tq.shape}")
print(f"  FP32 inference time:     {t_fp32:.3f} s")
print(f"  TQ inference time:       {t_tq:.3f} s")
print(f"  Output MSE (TQ vs FP32): {diff_mse:.6f}")
print(f"  Output distortion:       {output_distortion_db:.1f} dB")
print(f"  TQ output finite:        {'PASS' if torch.isfinite(pred_tq).all() else 'FAIL'}")

## 6. NMSE Evaluation (Synthetic Dataset)

In [ ]:
class SyntheticChannelDataset(torch.utils.data.Dataset):
    def __init__(self, n=128, seed=42):
        torch.manual_seed(seed)
        self.smomps    = torch.randn(n, 32, 4, 576) * 0.1
        self.accurates = torch.randn(n, 32, 4, 576) * 0.1
        self.rss_maps  = torch.rand(n, 2, 64, 64) * 2 - 1
    def __len__(self): return len(self.smomps)
    def __getitem__(self, i):
        return self.smomps[i], self.accurates[i], self.rss_maps[i]


def compute_nmse_db(pred, target):
    n = pred.shape[1] // 2
    pred_c   = torch.complex(pred[:, :n],   pred[:, n:])
    target_c = torch.complex(target[:, :n], target[:, n:])
    mse      = torch.mean(torch.abs(pred_c - target_c) ** 2)
    power    = torch.mean(torch.abs(target_c) ** 2).clamp(min=1e-12)
    return 10 * np.log10((mse / power).item() + 1e-12)


@torch.no_grad()
def run_eval(model, loader, device, label):
    model.eval().to(device)
    nmses, t0 = [], time.perf_counter()
    for smomp, accurate, rss in loader:
        smomp, accurate, rss = smomp.to(device), accurate.to(device), rss.to(device)
        pred = model(smomp, rss)
        nmses.append(compute_nmse_db(pred, accurate))
    elapsed = time.perf_counter() - t0
    avg = np.mean(nmses)
    print(f"  [{label}] NMSE={avg:.2f} dB  time={elapsed:.2f} s  ({len(loader.dataset)/elapsed:.1f} samples/s)")
    return avg, nmses


dataset = SyntheticChannelDataset(n=64)
loader  = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False)

print("=== NMSE Evaluation (synthetic, random weights) ===")
avg_fp32, nmses_fp32 = run_eval(model_fp32, loader, DEVICE, "FP32  ")
avg_tq,   nmses_tq   = run_eval(model_q,    loader, DEVICE, "TQ-3b ")

delta = avg_tq - avg_fp32
print(f"\n  NMSE delta (TQ - FP32): {delta:+.3f} dB")
if abs(delta) < 1.0:
    print("  STATUS: PASS — <1 dB degradation (acceptable PTQ)")
elif abs(delta) < 3.0:
    print("  STATUS: MARGINAL — consider QAT fine-tuning")
else:
    print("  STATUS: WARN — >3 dB (expected with random weights; test with real checkpoint)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(nmses_fp32, bins=15, alpha=0.7, label=f'FP32 (mean={avg_fp32:.1f} dB)')
axes[0].hist(nmses_tq,   bins=15, alpha=0.7, label=f'TQ-3bit (mean={avg_tq:.1f} dB)')
axes[0].set_xlabel('NMSE (dB)')
axes[0].set_ylabel('count')
axes[0].set_title('NMSE distribution (synthetic dataset)')
axes[0].legend()

axes[1].scatter(nmses_fp32, nmses_tq, s=30, alpha=0.7)
mn = min(min(nmses_fp32), min(nmses_tq))
mx = max(max(nmses_fp32), max(nmses_tq))
axes[1].plot([mn, mx], [mn, mx], 'r--', label='parity')
axes[1].set_xlabel('FP32 NMSE (dB)')
axes[1].set_ylabel('TQ-3bit NMSE (dB)')
axes[1].set_title('FP32 vs TQ-3bit per-batch NMSE')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Layer-by-Layer Reconstruction Quality

In [ ]:
fp32_modules = dict(model_fp32.named_modules())
layer_data = []

for name, m_tq in model_q.named_modules():
    if not isinstance(m_tq, (TQLinear, TQConv2d, TQConvTranspose2d)):
        continue
    m_fp32 = fp32_modules.get(name)
    if m_fp32 is None or not hasattr(m_fp32, 'weight'):
        continue
    W_orig = m_fp32.weight.data
    W_hat  = m_tq.tq.decompress(dtype=W_orig.dtype)
    mse    = ((W_hat - W_orig) ** 2).mean().item()
    var    = W_orig.var().item()
    ratio  = mse / max(var, 1e-12)
    layer_data.append((name, type(m_tq).__name__, ratio, W_orig.numel(), m_tq.tq.compression_ratio()))

print(f"TQ layers analyzed: {len(layer_data)}")
print(f"{'Name':50s} {'Type':20s} {'MSE/Var':10s} {'Params':10s} {'Comp':6s}")
print("-" * 100)
for name, ltype, ratio, numel, comp in layer_data[:20]:
    print(f"{name:50s} {ltype:20s} {ratio:.4f}     {numel:8,d}   {comp:.1f}x")
if len(layer_data) > 20:
    print(f"  ... and {len(layer_data)-20} more layers")

In [ ]:
if layer_data:
    ratios = [d[2] for d in layer_data]
    numels = [d[3] for d in layer_data]
    comps  = [d[4] for d in layer_data]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(ratios, bins=30)
    axes[0].axvline(LLOYD_MAX_3BIT_DISTORTION_RATIO, color='r', linestyle='--',
                    label=f'theoretical {LLOYD_MAX_3BIT_DISTORTION_RATIO:.4f}')
    axes[0].set_xlabel('MSE/Var per layer')
    axes[0].set_ylabel('count')
    axes[0].set_title('Reconstruction quality distribution')
    axes[0].legend()

    axes[1].scatter(numels, ratios, s=20, alpha=0.6)
    axes[1].set_xlabel('Weight tensor size (params)')
    axes[1].set_ylabel('MSE/Var')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title('MSE/Var vs layer size')

    axes[2].hist(comps, bins=15)
    axes[2].set_xlabel('Compression ratio (x)')
    axes[2].set_ylabel('count')
    axes[2].set_title('Compression ratio distribution')

    plt.tight_layout()
    plt.show()

    print(f"MSE/Var — median: {np.median(ratios):.4f}  max: {max(ratios):.4f}  min: {min(ratios):.4f}")
    print(f"Comp    — median: {np.median(comps):.1f}x  max: {max(comps):.1f}x  min: {min(comps):.1f}x")

## 8. (Optional) Real Checkpoint Evaluation

In [ ]:
# Upload a real checkpoint, then uncomment and set CHECKPOINT_PATH

# CHECKPOINT_PATH = "/content/best_model.pth"

# if os.path.exists(CHECKPOINT_PATH):
#     ckpt  = torch.load(CHECKPOINT_PATH, map_location="cpu")
#     state = ckpt.get("model_state_dict", ckpt)
#     model_fp32_ckpt = ImprovedPhysicsInformedUNet(
#         channel_shape=(32, 4, 576), rss_size=64, use_dbm_values=True
#     ).eval()
#     model_fp32_ckpt.load_state_dict(state, strict=False)
#     model_q_ckpt = quantize_model(model_fp32_ckpt, block_dim_linear=128, block_dim_conv=64)
#     print(model_size_report(model_fp32_ckpt, model_q_ckpt))
#     avg_fp32_ckpt, _ = run_eval(model_fp32_ckpt, loader, DEVICE, "FP32 (ckpt)")
#     avg_tq_ckpt,   _ = run_eval(model_q_ckpt,    loader, DEVICE, "TQ-3b (ckpt)")
#     print(f"  NMSE delta: {avg_tq_ckpt - avg_fp32_ckpt:+.3f} dB")
# else:
#     print("No checkpoint found — skipping.")

print("(Disabled — set CHECKPOINT_PATH above to enable.)")

## 9. PG-KD Training Plots

Reads `training_history_{preset}.json` files produced by `train_kd.py` and renders:
- Validation NMSE (dB) over epochs — all three presets overlaid
- Total training loss over epochs
- Per-component loss breakdown (NMSE, KD-soft, Physics, Xattn)
- Learning-rate schedule

**Set `CHECKPOINT_DIR` to wherever your `.json` history files live.**

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── Configure path ────────────────────────────────────────────────────────────
# Local run:
CHECKPOINT_DIR = Path("experiment2_PhysicsInformedKnowledgeDistillation/checkpoints")
# Colab (after Drive mount):
# CHECKPOINT_DIR = Path("/content/drive/MyDrive/ECE228/checkpoints")

PRESETS = ["light", "moderate", "extreme"]
COLORS  = {"light": "#2196F3", "moderate": "#4CAF50", "extreme": "#F44336"}
LABELS  = {"light": "Light (10×, 36M)", "moderate": "Moderate (20×, 18M)", "extreme": "Extreme (39×, 9.3M)"}

# ── Load histories ────────────────────────────────────────────────────────────
histories: dict = {}
for preset in PRESETS:
    p = CHECKPOINT_DIR / f"training_history_{preset}.json"
    if p.exists():
        with open(p) as f:
            data = json.load(f)
        histories[preset] = data["history"]
        print(f"  Loaded {preset}: {len(histories[preset])} epochs")
    else:
        print(f"  [{preset}] not found: {p}")

if not histories:
    print("\nNo history files found. Train at least one preset first, or check CHECKPOINT_DIR.")

In [ ]:
if not histories:
    print("Nothing to plot — run the cell above first.")
else:
    def _get(h, key):
        return [e[key] for e in h]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("PG-KD Training History — All Presets", fontsize=14, fontweight="bold", y=1.01)

    # ── (0,0) Val NMSE ────────────────────────────────────────────────────────
    ax = axes[0, 0]
    for preset, h in histories.items():
        epochs    = _get(h, "epoch")
        val_nmse  = _get(h, "val_nmse")
        best_ep   = epochs[int(np.argmin(val_nmse))]
        best_val  = min(val_nmse)
        ax.plot(epochs, val_nmse, color=COLORS[preset], lw=2, label=LABELS[preset])
        ax.axvline(best_ep, color=COLORS[preset], lw=1, ls="--", alpha=0.6)
        ax.annotate(f"{best_val:.2f} dB", xy=(best_ep, best_val),
                    xytext=(4, 4), textcoords="offset points",
                    fontsize=8, color=COLORS[preset])
    ax.set_title("Validation NMSE (dB)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NMSE (dB)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── (0,1) Total Train Loss ────────────────────────────────────────────────
    ax = axes[0, 1]
    for preset, h in histories.items():
        ax.plot(_get(h, "epoch"), _get(h, "total"),
                color=COLORS[preset], lw=2, label=LABELS[preset])
    ax.set_title("Total Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── (0,2) Learning Rate ───────────────────────────────────────────────────
    ax = axes[0, 2]
    for preset, h in histories.items():
        ax.plot(_get(h, "epoch"), _get(h, "lr"),
                color=COLORS[preset], lw=2, label=LABELS[preset])
    ax.set_title("Learning Rate Schedule")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("LR")
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2e"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── (1,0) NMSE Loss Component ─────────────────────────────────────────────
    ax = axes[1, 0]
    for preset, h in histories.items():
        ax.plot(_get(h, "epoch"), _get(h, "nmse"),
                color=COLORS[preset], lw=2, label=LABELS[preset])
    ax.set_title(r"$\mathcal{L}_{NMSE}$ (Ground-Truth Term)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── (1,1) KD-Soft Loss ────────────────────────────────────────────────────
    ax = axes[1, 1]
    for preset, h in histories.items():
        ax.plot(_get(h, "epoch"), _get(h, "kd_soft"),
                color=COLORS[preset], lw=2, label=LABELS[preset])
    ax.set_title(r"$\mathcal{L}_{KD}^{soft}$ (Teacher Imitation)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── (1,2) Xattn Feature Loss ──────────────────────────────────────────────
    ax = axes[1, 2]
    for preset, h in histories.items():
        ax.plot(_get(h, "epoch"), _get(h, "xattn_feat"),
                color=COLORS[preset], lw=2, label=LABELS[preset])
    ax.set_title(r"$\mathcal{L}_{xattn}$ (Feature Alignment)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("kd_training_plots.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → kd_training_plots.png")

In [ ]:
# ── Final NMSE bar chart (best checkpoint per preset) ─────────────────────────
if histories:
    best_nmse = {preset: min(_get(h, "val_nmse")) for preset, h in histories.items()}

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(
        [LABELS[p] for p in best_nmse],
        [best_nmse[p] for p in best_nmse],
        color=[COLORS[p] for p in best_nmse],
        edgecolor="black", linewidth=0.8, width=0.5,
    )
    for bar, preset in zip(bars, best_nmse):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                f"{best_nmse[preset]:.2f} dB",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax.set_title("Best Validation NMSE per Student Preset", fontsize=12, fontweight="bold")
    ax.set_ylabel("NMSE (dB)")
    ax.set_xlabel("Student Model")
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(min(best_nmse.values()) - 1, max(best_nmse.values()) + 1)
    plt.tight_layout()
    plt.savefig("kd_best_nmse_bar.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → kd_best_nmse_bar.png")

## Summary

| Test | What it checks |
|------|----------------|
| FHT H²=I | Walsh-Hadamard is self-inverse |
| Rotation invertibility | `inverse_rotation(random_rotation(x))` ≈ x |
| Lloyd-Max distortion | MSE/Var ≈ 0.0194 for N(0,1) input |
| Outlier spreading | Hadamard distributes energy — key for quantization quality |
| TurboQuantTensor roundtrip | compress then decompress ≈ original |
| Compression ratio | ~10.2x vs FP32 at block_dim=128 |
| TQLinear / TQConv2d | Drop-in wrappers produce near-FP32 output |
| PINN model quantization | All Linear/Conv layers replaced, forward runs |
| NMSE evaluation | TQ vs FP32 NMSE delta on synthetic data |
| Layer MSE | Per-layer reconstruction quality distribution |